# Feature Selectivity Score (FSS) — TabPFN v2

Per-head feature-attention analysis across three sklearn datasets:
**breast_cancer** (2 classes), **wine** (3 classes), **digits** (10 classes).

**Pipeline**
1. Fit `TabPFNClassifier(n_estimators=1)` on each dataset and instrument it.
2. Capture per-head feature attention; compute the FSS matrix $(L \times H)$.
3. Plot the FSS heatmap (one panel per dataset).
4. Rank heads by FSS; print top-3 / bottom-3 per dataset.
5. Visualise top-3 and bottom-3 FSS attention maps per dataset.
6. Progressive FSS-ranked ablation for $N \in \{5, 10, \dots, 70, 71, 72\}$.
7. Overlay ablation curves across the three datasets.
8. Permutation test: per-dataset top-35 FSS heads vs. random 35 feature heads
   sampled from the non-aligned pool (feature attention only).


In [ ]:
# Setup, dataset loading, fitting and capturing attention using InstrumentedTabPFN 

import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import entropy as sp_entropy
from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from tabpfn import TabPFNClassifier
from instrumented_tabpfn import InstrumentedTabPFN

N_TRAIN = 64; N_TEST = 32; SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

DS_ORDER = ['breast_cancer', 'wine', 'digits']

def _auc(y_true, proba):
    """Macro-averaged OvR ROC-AUC for multi-class, plain AUC for binary."""
    if proba.shape[1] == 2:
        return roc_auc_score(y_true, proba[:, 1])
    return roc_auc_score(y_true, proba, multi_class='ovr', average='macro')

def _load_raw():
    Xb, yb = load_breast_cancer(return_X_y=True)
    Xw, yw = load_wine(return_X_y=True)
    Xd, yd = load_digits(return_X_y=True)
    return {'breast_cancer': (Xb, yb, load_breast_cancer().feature_names),
            'wine':          (Xw, yw, load_wine().feature_names),
            'digits':        (Xd, yd, np.array([f'pix_{i}' for i in range(Xd.shape[1])]))}

splits = {}
for name, (X, y, fnames) in _load_raw().items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, train_size=N_TRAIN, test_size=N_TEST,
        random_state=SEED, stratify=y,
    )
    sc = StandardScaler().fit(X_tr)
    splits[name] = {
        'X_tr': sc.transform(X_tr), 'X_te': sc.transform(X_te),
        'y_tr': y_tr, 'y_te': y_te,
        'feature_names': fnames,
    }
    print(f"  {name:<14s} n_tr={N_TRAIN} n_te={N_TEST} feat={X_tr.shape[1]} classes={len(np.unique(y))}")

ds_state = {}      # ds -> dict with clf, inst, A_mean, NL, NH, C
for ds in DS_ORDER:
    s = splits[ds]
    print(f"\n[{ds}] fitting + capturing …")
    clf = TabPFNClassifier(n_estimators=1, device='cpu', random_state=SEED)
    clf.fit(s['X_tr'], s['y_tr'])
    inst = InstrumentedTabPFN(clf.model_)

    NL = len(clf.model_.blocks)
    NH = clf.model_.blocks[0].per_column_attention_between_cells.num_heads

    inst.enable_capture()
    clf.predict_proba(s['X_te'])
    caps = inst.get_captured()
    inst.disable_capture()

    A_mean = {li: caps['feature'][li].detach().cpu().mean(dim=0).numpy()
              for li in range(NL)}
    C = A_mean[0].shape[-1]

    ds_state[ds] = {
        'clf': clf, 'inst': inst, 'A_mean': A_mean,
        'NL': NL, 'NH': NH, 'C': C,
    }
    print(f"  layers={NL}  heads={NH}  cols(C)={C}  total_heads={NL*NH}")


In [ ]:
# For each head h, the column attention profile is the average attention
# weight received by each key column (averaged across all query rows):
#   col_profile = A[h].mean(axis=0)  →  shape (C,)
#
# FSS = 1 - entropy(col_profile) / log(C)
#   FSS = 0 → uniform (diffuse)
#   FSS = 1 → all mass on one column (peaked)
#

for ds in DS_ORDER:
    st = ds_state[ds]
    NL, NH, C = st['NL'], st['NH'], st['C']
    fss_matrix       = np.zeros((NL, NH))
    top_col_matrix   = np.zeros((NL, NH), dtype=int)
    tgt_share_matrix = np.zeros((NL, NH))
    max_H = np.log(C)

    for li in range(NL):
        for hi in range(NH):
            A = st['A_mean'][li][hi]
            cp = A.mean(axis=0)
            cp = cp / cp.sum()
            fss_matrix[li, hi]       = 1.0 - sp_entropy(cp) / max_H
            top_col_matrix[li, hi]   = int(np.argmax(cp))
            tgt_share_matrix[li, hi] = cp[C - 1]

    st['fss_matrix']       = fss_matrix
    st['top_col_matrix']   = top_col_matrix
    st['tgt_share_matrix'] = tgt_share_matrix

    n_target = (top_col_matrix == C - 1).sum()
    print(f"[{ds}]  FSS range=[{fss_matrix.min():.3f}, {fss_matrix.max():.3f}]  "
          f"mean={fss_matrix.mean():.3f}  "
          f"target-attending heads={n_target}/{NL*NH}")


In [ ]:
# FSS heatmaps — three datasets 

fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(4 * len(DS_ORDER), 14), squeeze=False)

for idx, ds in enumerate(DS_ORDER):
    st = ds_state[ds]
    NL, NH = st['NL'], st['NH']
    M = st['fss_matrix']
    ax = axes[0, idx]
    im = ax.imshow(M, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    for li in range(NL):
        for hi in range(NH):
            v = M[li, hi]
            color = 'white' if v < 0.4 else 'black'
            ax.text(hi, li, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color=color, fontweight='bold')
    ax.set_xticks(range(NH))
    ax.set_xticklabels([f'H{h}' for h in range(NH)])
    ax.set_yticks(range(NL))
    ax.set_yticklabels([f'L{l:02d}' for l in range(NL)], fontsize=8)
    ax.set_xlabel('Head')
    if idx == 0:
        ax.set_ylabel('Layer')
    ax.set_title(f'FSS — {ds}', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label='FSS')

fig.suptitle('Feature Selectivity Score (FSS)\n0 = uniform, 1 = peaked on one column',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.savefig('../figures/fss_heatmap_multi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Rank all heads by FSS per dataset 

SUMMARY_N = 3

for ds in DS_ORDER:
    st = ds_state[ds]
    NL, NH, C = st['NL'], st['NH'], st['C']
    fnames = splits[ds]['feature_names']

    ranked = []
    for li in range(NL):
        for hi in range(NH):
            tc = int(st['top_col_matrix'][li, hi])
            ranked.append({
                'layer': li, 'head': hi,
                'label': f'L{li:02d}H{hi}',
                'fss': float(st['fss_matrix'][li, hi]),
                'top_col': tc,
                'top_col_name': 'TARGET' if tc == C - 1 else (
                    fnames[tc] if tc < len(fnames) else f'col_{tc}'),
                'target_share': float(st['tgt_share_matrix'][li, hi]),
            })
    ranked.sort(key=lambda h: -h['fss'])
    st['ranked_heads'] = ranked

    print(f"\n=== {ds} — top-{SUMMARY_N} and bottom-{SUMMARY_N} heads by FSS ===")
    print(f"{'Rank':>4s}  {'Head':>6s}  {'FSS':>6s}  {'TopCol':>6s}  {'TgtShare':>8s}  TopColName")
    print('-' * 72)
    for i, h in enumerate(ranked[:SUMMARY_N]):
        print(f"{i+1:4d}  {h['label']:>6s}  {h['fss']:.4f}  "
              f"{h['top_col']:6d}  {h['target_share']:.4f}    {h['top_col_name']}")
    print('  ...')
    for i, h in enumerate(ranked[-SUMMARY_N:]):
        rank = len(ranked) - SUMMARY_N + i + 1
        print(f"{rank:4d}  {h['label']:>6s}  {h['fss']:.4f}  "
              f"{h['top_col']:6d}  {h['target_share']:.4f}    {h['top_col_name']}")


In [ ]:
# Top-3 and bottom-3 FSS attention maps per dataset 

TOP_N_VIS = 3
BOTTOM_N_VIS = 3

for ds in DS_ORDER:
    st = ds_state[ds]
    C = st['C']
    top    = st['ranked_heads'][:TOP_N_VIS]
    bottom = st['ranked_heads'][-BOTTOM_N_VIS:][::-1]   # lowest first

    fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
    vmax = max(st['A_mean'][h['layer']][h['head']].max() for h in (top + bottom))

    for col_idx, h in enumerate(top):
        ax = axes[0, col_idx]
        A = st['A_mean'][h['layer']][h['head']]
        im = ax.imshow(A, cmap='viridis', aspect='auto', vmin=0, vmax=vmax)
        ax.axvline(x=C - 1.5, color='red', lw=0.8, ls='--', alpha=0.7)
        ax.axhline(y=C - 1.5, color='red', lw=0.8, ls='--', alpha=0.7)
        ax.set_title(f"top {col_idx+1}: {h['label']}\n"
                     f"FSS={h['fss']:.3f}  top_col={h['top_col_name'][:14]}",
                     fontsize=9)
        ax.set_xticks([0, C // 2, C - 1])
        ax.set_xticklabels(['feat 0', f'feat {C // 2}', 'TGT'], fontsize=7)
        ax.set_yticks([])

    for col_idx, h in enumerate(bottom):
        ax = axes[1, col_idx]
        A = st['A_mean'][h['layer']][h['head']]
        im = ax.imshow(A, cmap='viridis', aspect='auto', vmin=0, vmax=vmax)
        ax.axvline(x=C - 1.5, color='red', lw=0.8, ls='--', alpha=0.7)
        ax.axhline(y=C - 1.5, color='red', lw=0.8, ls='--', alpha=0.7)
        ax.set_title(f"bottom {col_idx+1}: {h['label']}\n"
                     f"FSS={h['fss']:.3f}",
                     fontsize=9)
        ax.set_xticks([0, C // 2, C - 1])
        ax.set_xticklabels(['feat 0', f'feat {C // 2}', 'TGT'], fontsize=7)
        ax.set_yticks([])

    axes[0, 0].set_ylabel('top-3', fontsize=10)
    axes[1, 0].set_ylabel('bottom-3', fontsize=10)
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02,
                 label='attention')
    fig.suptitle(f'Feature attention — {ds}\n'
                 'Top row = highest FSS, bottom row = lowest FSS\n'
                 'Red dashed line = target column boundary',
                 fontsize=12)
    plt.savefig(f'../figures/topbot_{ds}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# FSS-ranked ablation per dataset 
#   1. Baseline ROC-AUC (no heads masked).
#   2. For N in N_VALUES: zero the top-N FSS heads in feature attention only,
#      record ROC-AUC and delta.
#

N_VALUES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 71, 72]

for ds in DS_ORDER:
    st = ds_state[ds]
    s = splits[ds]
    inst = st['inst']

    inst.clear_masks()
    proba_base = st['clf'].predict_proba(s['X_te'])
    base_auc = _auc(s['y_te'], proba_base)
    st['baseline_auc'] = base_auc
    print(f"\n[{ds}] baseline ROC-AUC = {base_auc:.4f}")

    res = []
    for N in N_VALUES:
        heads = [(h['layer'], h['head']) for h in st['ranked_heads'][:N]]
        inst.set_masks({(li, hi, 'feature'): True for (li, hi) in heads})
        proba = st['clf'].predict_proba(s['X_te'])
        auc = _auc(s['y_te'], proba)
        drop = base_auc - auc
        res.append({'N': N, 'auc': auc, 'drop': drop})
        print(f"  drop top-{N:2d}:  AUC={auc:.4f}  Δ={-drop:+.4f}")
        inst.clear_masks()

    st['ablation'] = res

    

In [ ]:
# Ablation curves 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

palette = {'breast_cancer': 'tab:red', 'wine': 'tab:green', 'digits': 'tab:purple'}
markers = {'breast_cancer': 'o',       'wine': 's',         'digits': 'D'}

# ── Left: raw ROC-AUC vs N ───────────────────────────────────────────────────
for ds in DS_ORDER:
    st = ds_state[ds]
    ns   = [0] + [r['N'] for r in st['ablation']]
    aucs = [st['baseline_auc']] + [r['auc'] for r in st['ablation']]
    ax1.plot(ns, aucs, f"{markers[ds]}-", color=palette[ds], lw=2, markersize=7,
             label=f"{ds} (base={st['baseline_auc']:.3f})")
    ax1.axhline(st['baseline_auc'], color=palette[ds], ls=':', lw=0.8, alpha=0.5)

ax1.set_xlabel('Number of feature heads dropped (FSS-ranked)', fontsize=11)
ax1.set_ylabel('ROC-AUC', fontsize=11)
ax1.set_title('FSS-ranked ablation — raw ROC-AUC', fontsize=12)
ax1.set_xticks([0] + N_VALUES)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=10, loc='lower left')
ax1.grid(axis='y', alpha=0.3)

# ── Right: drop from baseline ────────────────────────────────────────────────
for ds in DS_ORDER:
    st = ds_state[ds]
    ns    = [0] + [r['N'] for r in st['ablation']]
    drops = [0.0] + [r['drop'] for r in st['ablation']]   # positive = degradation
    ax2.plot(ns, drops, f"{markers[ds]}-", color=palette[ds], lw=2, markersize=7,
             label=ds)

ax2.axhline(0, color='grey', ls='--', lw=1)
ax2.set_xlabel('Number of feature heads dropped (FSS-ranked)', fontsize=11)
ax2.set_ylabel('ROC-AUC drop from baseline', fontsize=11)
ax2.set_title('FSS-ranked ablation — degradation curves', fontsize=12)
ax2.set_xticks([0] + N_VALUES)
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/fss_ablation_curves_multi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Permutation test — per-dataset top-35 FSS heads ──────────────────
# For each dataset:
#   1. baseline_auc = AUC with no heads masked.
#   2. obs_drop  = AUC(top-N_DROP FSS heads masked) − baseline_auc.
#   3. Null distribution: N_PERM random N_DROP-head sets drawn (without
#      replacement) from the NON-ALIGNED pool — i.e. only feature heads NOT in
#      the observed set. This guarantees every random draw is disjoint from
#      the observed set, giving a stricter null than sampling all 72 heads.
#   4. Report null mean ± std, z-score, and one-tailed empirical p-value
#      (fraction of null runs at least as bad as the observed drop).
#
# Reproducibility: Python `random` and torch RNG are seeded.

import random as _pyrandom

N_DROP    = 35
N_PERM    = 50
SEED_PERM = 0
ATTN_TYPE = 'feature'

_pyrandom.seed(SEED_PERM)
torch.manual_seed(SEED_PERM)

print(f"{'='*64}")
print(f"  PERMUTATION TEST  —  per-dataset top-{N_DROP} FSS heads  (feature attention)")
print(f"  N_PERM={N_PERM}   pool=non_aligned (sample from heads NOT in fixed set)")
print(f"  metric: roc_auc (binary or macro-OvR)")
print(f"{'='*64}\n")

perm_results = {}

for ds in DS_ORDER:
    st = ds_state[ds]
    s = splits[ds]
    inst = st['inst']
    NL, NH = st['NL'], st['NH']

    # Observed set: this dataset's own top-N_DROP FSS heads
    fixed_heads        = [(h['layer'], h['head']) for h in st['ranked_heads'][:N_DROP]]
    fixed_heads_labels = [f'L{li:02d}H{hi}' for (li, hi) in fixed_heads]

    all_heads   = [(li, hi) for li in range(NL) for hi in range(NH)]
    fixed_set   = set(fixed_heads)
    non_aligned = [h for h in all_heads if h not in fixed_set]

    # 1. baseline
    inst.clear_masks()
    base_auc = _auc(s['y_te'], st['clf'].predict_proba(s['X_te']))

    # 2. observed
    inst.set_masks({(li, hi, ATTN_TYPE): True for (li, hi) in fixed_heads})
    obs_auc  = _auc(s['y_te'], st['clf'].predict_proba(s['X_te']))
    obs_drop = obs_auc - base_auc
    inst.clear_masks()

    # 3. null
    null_drops = np.empty(N_PERM)
    for k in range(N_PERM):
        rand_heads = _pyrandom.sample(non_aligned, N_DROP)
        inst.set_masks({(li, hi, ATTN_TYPE): True for (li, hi) in rand_heads})
        null_drops[k] = _auc(s['y_te'], st['clf'].predict_proba(s['X_te'])) - base_auc
        inst.clear_masks()

    null_mean = null_drops.mean()
    null_std  = null_drops.std()
    z_score   = (obs_drop - null_mean) / (null_std + 1e-9)
    p_emp     = float((null_drops <= obs_drop).mean())
    sig       = '(significant)' if p_emp < 0.05 else '(not significant)'

    perm_results[ds] = {
        'base_auc': base_auc, 'obs_auc': obs_auc, 'obs_drop': obs_drop,
        'null_drops': null_drops,
        'null_mean': null_mean, 'null_std': null_std,
        'z': z_score, 'p_emp': p_emp,
        'fixed_heads_labels': fixed_heads_labels,
    }

    print(f"  {ds}")
    print(f"    head set      : " + ', '.join(fixed_heads_labels))
    print(f"    baseline AUC  : {base_auc:.4f}")
    print(f"    observed AUC  : {obs_auc:.4f}")
    print(f"    observed drop : {obs_drop:+.4f}")
    print(f"    null mean     : {null_mean:+.4f} ± {null_std:.4f}")
    print(f"    z-score       : {z_score:+.2f}")
    print(f"    p-value       : {p_emp:.3f}  {sig}\n")

# ── Histogram: null distribution + observed marker, one panel per dataset ───
fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(5.2 * len(DS_ORDER), 4.5),
                         squeeze=False)
for idx, ds in enumerate(DS_ORDER):
    r = perm_results[ds]
    ax = axes[0, idx]
    ax.hist(r['null_drops'], bins=15, color='#4a90d9', alpha=0.75,
            edgecolor='white', label=f'null (n={N_PERM})')
    ax.axvline(r['obs_drop'], color='red', lw=2,
               label=f"observed = {r['obs_drop']:+.4f}")
    ax.axvline(r['null_mean'], color='black', ls='--', lw=1.2,
               label=f"null mean = {r['null_mean']:+.4f}")
    ax.set_title(f"{ds}\nz = {r['z']:+.2f}   p = {r['p_emp']:.3f}  "
                 f"({'sig.' if r['p_emp'] < 0.05 else 'n.s.'})", fontsize=11)
    ax.set_xlabel('drop = AUC(ablated) − AUC(baseline)', fontsize=10)
    if idx == 0:
        ax.set_ylabel('null run count', fontsize=10)
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)

fig.suptitle(f"Permutation test — per-dataset top-{N_DROP} FSS heads "
             f"vs. random {N_DROP}-head samples from the non-aligned pool "
             f"(feature attention)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../figures/fss_permutation_test_multi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
